# Windowed `aggregate=` — a monthly NDVI series

`download(aggregate=AggregationConfig(freq='1MS', op='mean'))` loops one render per month, writing one GeoTIFF per window named `{key}_{freq}_{YYYYMMDD}.tif` (the same shape as the ECMWF / CMEMS backends).

## Credentials guard

In [ ]:
import os
from pathlib import Path


def has_sh_credentials() -> bool:
    """Whether Sentinel Hub OAuth client-credentials are available."""
    return bool(os.environ.get("SH_CLIENT_ID") and os.environ.get("SH_CLIENT_SECRET"))


def run_or_skip(facade, **download_kwargs):
    """Run the live download when credentials exist, else print a skip note.

    Keeps the notebook executing top-to-bottom with no errors whether or not
    Sentinel Hub credentials are configured (so the docs build never needs
    secrets). Set SH_CLIENT_ID / SH_CLIENT_SECRET to run the cell for real.
    """
    if not has_sh_credentials():
        print(
            "No Sentinel Hub credentials found - skipping the live download.\n"
            "Mint an OAuth client_credentials pair in the CDSE Dashboard and set\n"
            "SH_CLIENT_ID / SH_CLIENT_SECRET (see the Authentication page)."
        )
        return []
    results = facade.download(**download_kwargs)
    for item in results:
        print(item)
    return results


## A 3-month NDVI series

In [ ]:
from earthlens import EarthLens
from earthlens.aggregate import AggregationConfig

facade = EarthLens(
    data_source='sentinel-hub',
    variables={'sentinel-2-l2a-ndvi': []},
    start='2020-04-01', end='2020-06-30',
    lat_lim=[40.80, 40.83], lon_lim=[14.24, 14.27],
    path='data/sh-monthly', resolution=20,
)
paths = run_or_skip(facade, aggregate=AggregationConfig(freq='1MS', op='mean'))
paths